# Project Skeleton

This notebook is a starting point. Fill in dataset-specific details (target column name, feature columns, evaluation metrics) before running end-to-end.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Paths and general config
DATA_DIR = Path("data")
LR_TRAIN_PATH = DATA_DIR / "lr_train.csv"
HR_TRAIN_PATH = DATA_DIR / "hr_train.csv"
LR_TEST_PATH = DATA_DIR / "lr_test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

RANDOM_SEED = 42
TARGET_COL = "target"  # TODO: replace with actual target column name
ID_COL = "id"          # TODO: replace with actual id column if present

np.random.seed(RANDOM_SEED)


In [ ]:
def load_datasets():
    """Load all provided datasets."""
    lr_train = pd.read_csv(LR_TRAIN_PATH)
    hr_train = pd.read_csv(HR_TRAIN_PATH)
    lr_test = pd.read_csv(LR_TEST_PATH)
    sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
    return lr_train, hr_train, lr_test, sample_submission


def train_validation_split(df: pd.DataFrame):
    """Split provided dataframe into train/validation. Update target column name as needed."""
    if TARGET_COL not in df.columns:
        raise ValueError(f"Missing target column '{TARGET_COL}'. Update TARGET_COL before splitting.")
    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]
    return train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, shuffle=True)


In [ ]:
# Load datasets and quick sanity checks
lr_train, hr_train, lr_test, sample_submission = load_datasets()

print("lr_train shape:", lr_train.shape)
print("hr_train shape:", hr_train.shape)
print("lr_test shape:", lr_test.shape)
print("sample_submission shape:", sample_submission.shape)

# Peek at the first few rows
lr_train.head(), hr_train.head(), lr_test.head(), sample_submission.head()

In [ ]:
# Identify feature types
NUMERIC_COLS: list[str] = []   # TODO: fill with numeric column names
CATEGORICAL_COLS: list[str] = []  # TODO: fill with categorical column names

# Preprocessing pipelines
numeric_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, NUMERIC_COLS),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
    ],
    remainder="drop",  # change to "passthrough" if you want to keep unused columns
)


In [ ]:
def build_model():
    """Return a baseline model wrapped in the preprocessing pipeline."""
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", model),
        ]
    )


# Train/validation split using the low-resolution training set by default
X_train, X_valid, y_train, y_valid = train_validation_split(lr_train)

# Fit and evaluate baseline
baseline_model = build_model()
baseline_model.fit(X_train, y_train)

valid_preds = baseline_model.predict(X_valid)
rmse = mean_squared_error(y_valid, valid_preds, squared=False)
print(f"Validation RMSE: {rmse:.4f}")

In [ ]:
# Fit on full training data (after you're satisfied with validation)
final_model = build_model()
final_model.fit(lr_train.drop(columns=[TARGET_COL]), lr_train[TARGET_COL])

# Predict on lr_test and prepare submission
if ID_COL not in lr_test.columns:
    raise ValueError(f"Missing id column '{ID_COL}'. Update ID_COL before creating submission.")

test_predictions = final_model.predict(lr_test)
submission = pd.DataFrame({
    ID_COL: lr_test[ID_COL],
    TARGET_COL: test_predictions,
})

SUBMISSION_PATH = Path("submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved submission to {SUBMISSION_PATH.resolve()}")